# 🦀 Day 2: Advanced macro_rules! — Repetition, TT Munchers
---

Today we go deeper into **macro_rules!** with repetition, nested macros, and advanced patterns.

- **Repetition**: `$($x:expr),*` and `$($x:expr),+`
- **Nested repetition**: `$( $key:expr => $val:expr ),*`
- **hashmap!** macro: `hashmap!{ "a" => 1, "b" => 2 }`
- **TT muncher pattern**: Processing token trees one at a time
- **Internal rules**: `@` prefix convention
- **Recursive macros** and `macro_export`

In [ ]:
// Repetition: $($x:expr),* (zero or more) and $($x:expr),+ (one or more)
macro_rules! sum {
    ($($x:expr),* $(,)?) => {
        {
            let mut total = 0;
            $( total += $x; )*
            total
        }
    };
}

println!("sum!(1, 2, 3) = {}", sum!(1, 2, 3));
println!("sum!(10) = {}", sum!(10));
println!("sum!() = {}", sum!());

In [ ]:
// hashmap! macro: nested repetition $( $key:expr => $val:expr ),*
use std::collections::HashMap;

macro_rules! hashmap {
    ($( $key:expr => $val:expr ),* $(,)?) => {
        {
            let mut m = HashMap::new();
            $( m.insert($key, $val); )*
            m
        }
    };
}

let m = hashmap! { "a" => 1, "b" => 2, "c" => 3 };
println!("hashmap: {:?}", m);

In [ ]:
// match_str! macro for string matching
macro_rules! match_str {
    ($s:expr, $($pat:literal => $result:expr),* $(,)?) => {
        {
            let mut out = None;
            $( if $s == $pat { out = Some($result); } )*
            out
        }
    };
}

let result = match_str!("hello", "hi" => 1, "hello" => 2, "hey" => 3);
println!("match_str result: {:?}", result);

In [ ]:
// TT muncher pattern: process token trees one at a time
// Internal rules with @ prefix
macro_rules! count_tts {
    (@accum () -> $count:expr) => { $count };
    (@accum ($first:tt $($rest:tt)*) -> $count:expr) => {
        count_tts!(@accum ($($rest)*) -> $count + 1)
    };
    ($($tts:tt)*) => {
        count_tts!(@accum ($($tts)*) -> 0)
    };
}

println!("count_tts!(a b c) = {}", count_tts!(a b c));
println!("count_tts!(1 2 3 4 5) = {}", count_tts!(1 2 3 4 5));

In [ ]:
// Recursive macro example
macro_rules! factorial {
    (0) => { 1 };
    ($n:expr) => { $n * factorial!($n - 1) };
}

println!("factorial!(5) = {}", factorial!(5));

## 📝 Exercise: struct_builder! macro

Build a `struct_builder!` macro that generates a struct with a builder.

Usage: `struct_builder!(Person { name: String, age: u32 })` should generate `Person`, `PersonBuilder`, and a `build()` method.

In [ ]:
// YOUR CODE HERE
// Design and implement struct_builder! macro
// Hint: use repetition for fields, generate builder methods

## 🎯 Key Takeaways

1. **Repetition**: `$($x:expr),*` matches zero-or-more; `$($x:expr),+` matches one-or-more
2. **Nested repetition**: Match key-value pairs with `$( $key:expr => $val:expr ),*`
3. **hashmap!** and similar macros use nested repetition for ergonomic initialization
4. **TT muncher**: Process `$($tt:tt)*` recursively with internal `@` rules
5. **Recursive macros** expand until a base case matches
6. `#[macro_export]` makes a macro available crate-wide when used in a library

---
**Tomorrow:** Debugging macros — cargo expand, tracing expansion